In [1]:
from PIL import Image
from pathlib import Path

In [2]:
def combine_images_1x2(
    left_path: str,
    right_path: str,
    out_path: str,
    align: str = "top",          # "top" | "center" | "bottom"
    bg_color=(255, 255, 255),    # white background
):
    """
    Combine two images into a 1-row, 2-column composite.

    - Keeps original pixel resolution
    - Pads shorter image vertically to match taller one
    - Saves to out_path (extension controls format: .png/.tif/.tiff)
    """
    left_path = Path(left_path)
    right_path = Path(right_path)
    out_path = Path(out_path)

    im1 = Image.open(left_path)
    im2 = Image.open(right_path)

    # Ensure consistent mode for TIFF/PNG mixing (avoid palette/alpha issues)
    im1 = im1.convert("RGB")
    im2 = im2.convert("RGB")

    w1, h1 = im1.size
    w2, h2 = im2.size

    out_w = w1 + w2
    out_h = max(h1, h2)

    canvas = Image.new("RGB", (out_w, out_h), color=bg_color)

    def y_offset(h):
        if align == "top":
            return 0
        if align == "bottom":
            return out_h - h
        # center
        return (out_h - h) // 2

    canvas.paste(im1, (0, y_offset(h1)))
    canvas.paste(im2, (w1, y_offset(h2)))

    # Preserve DPI metadata if present (common in TIFF)
    dpi = im1.info.get("dpi", None) or im2.info.get("dpi", None)

    save_kwargs = {}
    if dpi is not None:
        save_kwargs["dpi"] = dpi

    # TIFF compression (good for journals)
    if out_path.suffix.lower() in [".tif", ".tiff"]:
        save_kwargs["compression"] = "tiff_lzw"

    canvas.save(out_path, **save_kwargs)
    return out_path

In [3]:

# TIFF output (journal-ready)
combine_images_1x2(
    left_path="CO2_RE.tiff",
    right_path="CH4_RE.tiff",
    out_path="RE_CO2_CH4_per2026-02-23.tiff",
    align="top"
)

PosixPath('RE_CO2_CH4_per2026-02-23.tiff')